In [0]:
from pyspark.sql import functions as F

In [0]:
sales_df = spark.table("samples.bakehouse.sales_transactions")
sales_df.display()

# sales_df = spark.sql("select * from samples.bakehouse.sales_transactions")

In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
select customerID, count(transactionID) as no_of_transactions from sales
group by customerID;

In [0]:
# Find out the no.of transactions made by each customer

trans_df = (
    sales_df
    .groupBy("customerID")
    .agg(
        F.count("transactionID").alias("no_of_transactions")
    )
)
trans_df.display()

In [0]:
# Find out the total amount spent by each customer by their master card.

result_df = (
    sales_df
    .filter(F.col("paymentMethod") == "mastercard")
    .groupBy("customerID")
    .agg(
        F.sum("totalPrice").alias("totalAmountSpent")
    )
)
result_df.display()

In [0]:
result_df = (
    sales_df
    .groupBy("product")
    .agg(
        F.sum("quantity").alias("total_quantity_sold")
    )
    .orderBy(F.col("total_quantity_sold").desc())
)
result_df.display()

In [0]:
nyc_df = spark.table("samples.nyctaxi.trips")
nyc_df.display()

In [0]:
# Find out the no.of trips occured in the month of january 2016 by day wise

result_df = (
    nyc_df
    # .filter(F.col("tpep_pickup_datetime").between("2016-01-01", "2016-02-01"))
    .filter(
        (F.year(F.col("tpep_pickup_datetime")) == 2016) &
        (F.month(F.col("tpep_pickup_datetime")) == 1)
    )
    .groupBy(
        F.year(F.col("tpep_pickup_datetime")),  
        F.month(F.col("tpep_pickup_datetime")), 
        F.dayofmonth(F.col("tpep_pickup_datetime"))
    )
    .agg(
        F.count("*").alias("no_of_trips")
    )
    .select(
        F.col("year(tpep_pickup_datetime)").alias("year"),
        F.col("month(tpep_pickup_datetime)").alias("month"),
        F.col("dayofmonth(tpep_pickup_datetime)").alias("day"),
        F.col("no_of_trips")
    )

)

result_df.display()